# 01 — Quickstart + MARS metadata

<sup>(C) Copyright 2026- ECMWF and individual contributors. Licensed under the Apache Licence, Version 2.0. See NOTICE in the repository root.</sup>

**What you will learn in this notebook**

- Encode a 2D tensor with `tensogram.encode()` and decode it back round-trip.
- Attach **MARS metadata** under `base[i]["mars"]` and read it back with dict-style access.
- Use `tensogram.decode_metadata()` to peek at headers without touching any payload bytes.
- Understand the `base` / `_reserved_` / `_extra_` metadata layout introduced in v0.6.
- Compute cross-object common keys with `tensogram.compute_common()` (a utility; commonalities are **not** encoded on the wire).

**Prerequisites**: see [`README.md`](README.md). No OS-level C libraries are needed for this notebook.

## Setup

We use `matplotlib.Agg` to stay headless-safe (the same code runs in CI).

In [ ]:
import matplotlib

matplotlib.use("Agg")  # headless: same cell runs in CI and on your laptop

import matplotlib.pyplot as plt
import numpy as np

import tensogram

print(f"tensogram version: {tensogram.__version__ if hasattr(tensogram, '__version__') else '?'}")

## 1. The simplest possible round-trip

A Tensogram **message** is a self-contained blob of bytes that holds one or more N-dimensional tensors plus structured metadata. Here we encode a single 180×360 synthetic "temperature" field into a message, decode it, and verify we got identical bytes back.

In [ ]:
# Synthetic sine/cosine field shaped like a global 1-degree temperature grid.
# Real-world data usage is covered in notebooks 03 (GRIB) and 04 (NetCDF).
nlat, nlon = 180, 360
lats = np.linspace(-90, 90, nlat)
lons = np.linspace(-180, 180, nlon)
LAT, LON = np.meshgrid(lats, lons, indexing="ij")

temperature = (
    273.15
    + 30 * np.cos(np.deg2rad(LAT))            # warm tropics, cold poles
    - 10 * np.cos(np.deg2rad(2 * LON))        # zonal wavenumber-2 pattern
).astype(np.float32)

print(f"shape={temperature.shape}  dtype={temperature.dtype}  size={temperature.nbytes} bytes")

In [ ]:
# Encode: the descriptor tells tensogram what kind of tensor this is.
descriptor = {
    "type": "ntensor",
    "shape": list(temperature.shape),
    "dtype": "float32",
    "encoding": "none",          # no quantization
    "filter": "none",            # no byte rearrangement
    "compression": "none",       # no compression for this round
}

metadata = {}        # minimal global metadata

message = bytes(tensogram.encode(metadata, [(descriptor, temperature)]))

print(f"encoded message: {len(message):,} bytes (raw tensor was {temperature.nbytes:,})")

In [ ]:
# Decode: returns (metadata, list-of-(descriptor, numpy-array)).
meta_out, objects = tensogram.decode(message)
decoded_desc, decoded_array = objects[0]

print(f"got back {len(objects)} object(s)")
print(f"  shape={decoded_array.shape}  dtype={decoded_array.dtype}")
print(f"  xxh3 hash: {decoded_desc.hash['value'] if decoded_desc.hash else '(none)'}")

# Bit-identical round-trip:
assert np.array_equal(decoded_array, temperature)
print("\nround-trip OK")

In [ ]:
# Visualise what we just round-tripped.
fig, ax = plt.subplots(figsize=(9, 4))
im = ax.imshow(
    decoded_array,
    origin="lower",
    extent=[lons.min(), lons.max(), lats.min(), lats.max()],
    cmap="RdYlBu_r",
    aspect="auto",
)
ax.set_xlabel("longitude (deg)")
ax.set_ylabel("latitude (deg)")
ax.set_title("Decoded synthetic 2m-temperature-like field")
plt.colorbar(im, ax=ax, label="Kelvin")
plt.tight_layout()
plt.show()

## 2. Anatomy of a message

A Tensogram message starts with the ASCII magic `TENSOGRM` and ends with a known terminator. Let's peek at both.

In [ ]:
print(f"first 8 bytes (magic):      {message[:8]!r}")
print(f"last 8 bytes (terminator):  {message[-8:].hex()}")
print(f"total length:               {len(message):,} bytes")

# The wire format lays out a message like this:
#   [preamble] [header frames: index + hash + metadata]
#   [data object frames: one per tensor]
#   [footer frames: hash + index + metadata]
#   [postamble: first_footer_offset + end-of-file marker]
#
# See docs/src/format/wire-format.md for the full spec.

## 3. Attaching MARS metadata — `base[i]["mars"]`

ECMWF describes weather fields with **MARS keys** like `class=od, stream=oper, param=2t, date=20260404`. In Tensogram v2 every object has its own independent metadata entry at `base[i]` — there is no global/per-object split.

The encoder auto-populates `base[i]["_reserved_"]["tensor"]` with shape/strides/dtype for each object. Application metadata like MARS goes under any key of your choice — conventionally `"mars"`.

In [ ]:
metadata_with_mars = {
    "base": [
        {
            "mars": {
                "class": "od",
                "stream": "oper",
                "expver": "0001",
                "date": "20260404",
                "time": "0000",
                "step": "0",
                "levtype": "sfc",
                "param": "2t",
            },
        }
    ],
    # _extra_ is a free-form catch-all for message-level annotations.
    "_extra_": {"source": "synthetic demo", "notebook": "01_quickstart_and_mars"},
}

message_mars = bytes(
    tensogram.encode(metadata_with_mars, [(descriptor, temperature)])
)

# decode_metadata() reads only the headers — it does not touch any payload.
meta = tensogram.decode_metadata(message_mars)

print(f"version: {meta.version}")
print(f"base entries: {len(meta.base)}")
print(f"mars keys on object 0: {sorted(meta.base[0]['mars'].keys())}")
print(f"_reserved_.tensor: {meta.base[0]['_reserved_']['tensor']}")
print(f"_extra_: {dict(meta.extra)}")

### Dict-style access on `Metadata`

The `Metadata` object exposed by the Python bindings has a convenient dict-style lookup. It searches `base[i]` entries in order (skipping the `_reserved_` key) and falls back to `_extra_`.

In [ ]:
# meta["mars"] → the first base entry's mars sub-map:
mars = meta["mars"]
print(f"class={mars['class']}  param={mars['param']}  date={mars['date']}")

# meta["source"] → falls through to _extra_:
print(f"source  = {meta['source']}")

# missing keys raise KeyError, just like a dict:
try:
    _ = meta["does_not_exist"]
except KeyError as e:
    print(f"KeyError caught: {e}")

## 4. Multi-object messages and discovering common keys

A single Tensogram message can carry many tensors — for example, a temperature field *and* its land-sea mask. Every `base[i]` is **independent** (duplicate-by-design): if 20 objects all share the same `class=od`, the CBOR encoder stores `"class": "od"` 20 times.

If you want to know which keys happen to be shared across all objects, you can compute it post-hoc. The Rust library exposes a `compute_common()` utility for this — a tiny dozen-line function that's also trivial to reproduce in Python, so we include the recipe inline below. The key design point is that commonalities are **never** encoded on the wire: they are a software convenience.

In [ ]:
# Build a second field. To make `compute_common` do something interesting
# we tag both base entries with a few identical top-level keys and let the
# MARS sub-map differ (`param`). `compute_common` compares values at the
# **top level** of each entry — two dicts that differ in any nested key
# are treated as "not common", so `mars` itself will NOT appear in the
# common set.
lsm = np.random.default_rng(seed=1).random((nlat, nlon)).astype(np.float32)

multi_meta = {
    "base": [
        {
            "source": "synthetic",       # identical top-level value
            "experiment": "demo-001",    # identical top-level value
            "mars": {"class": "od", "stream": "oper", "date": "20260404", "param": "2t"},
        },
        {
            "source": "synthetic",
            "experiment": "demo-001",
            "mars": {"class": "od", "stream": "oper", "date": "20260404", "param": "lsm"},
        },
    ],
}

multi_desc_temp = {
    "type": "ntensor", "shape": list(temperature.shape), "dtype": "float32",
    "encoding": "none", "filter": "none", "compression": "none",
}
multi_desc_lsm = {
    "type": "ntensor", "shape": list(lsm.shape), "dtype": "float32",
    "encoding": "none", "filter": "none", "compression": "none",
}

multi_msg = bytes(tensogram.encode(
    multi_meta,
    [(multi_desc_temp, temperature), (multi_desc_lsm, lsm)],
))
print(f"multi-object message: {len(multi_msg):,} bytes, 2 objects")

meta_multi = tensogram.decode_metadata(multi_msg)
for i, entry in enumerate(meta_multi.base):
    print(f"  base[{i}].mars = {entry['mars']}")

In [ ]:
def compute_common(base_entries):
    """Return a dict of keys that appear with identical values in every entry.

    Ignores the library-managed ``_reserved_`` key (which always differs
    per object because it carries shape/strides). Mirrors the Rust
    ``tensogram::compute_common`` semantics.
    """
    if not base_entries:
        return {}
    common = {}
    for key, value in base_entries[0].items():
        if key == "_reserved_":
            continue
        if all(entry.get(key) == value for entry in base_entries[1:]):
            common[key] = value
    return common


common = compute_common(meta_multi.base)
print(f"common keys across all base entries: {common}")

# The two identical top-level keys appear in the common set:
assert common == {"source": "synthetic", "experiment": "demo-001"}

# `mars` is NOT in the common set — its sub-map differs by `param`.
# This is by design: the library compares top-level values only.
# If you need to discover commonalities inside nested maps, recurse
# explicitly or flatten the metadata before encoding.
assert "mars" not in common

## 5. Metadata-only reads

`tensogram.decode_metadata()` reads only the header and footer frames — it never touches a payload byte. This matters when you're scanning a large `.tgm` file to find the specific object you want.

In [ ]:
import time

start = time.perf_counter()
meta_only = tensogram.decode_metadata(multi_msg)
metadata_ms = (time.perf_counter() - start) * 1000

start = time.perf_counter()
_meta, _objs = tensogram.decode(multi_msg)
full_ms = (time.perf_counter() - start) * 1000

print(f"decode_metadata only: {metadata_ms:.2f} ms")
print(f"full decode:          {full_ms:.2f} ms")
print(f"speedup:              {full_ms / metadata_ms:.1f}x")

## Where to go next

- [`02_encoding_and_fidelity.ipynb`](02_encoding_and_fidelity.ipynb) — understand the encoding pipeline and choose a compressor.
- [`03_from_grib_to_tensogram.ipynb`](03_from_grib_to_tensogram.ipynb) — do the same thing with **real** ECMWF opendata GRIB.
- [`../../docs/src/concepts/messages.md`](../../docs/src/concepts/messages.md) — the full wire-format reference.